# SOC-Env: TRL GRPO Training with Unsloth (Qwen2.5-7B-Instruct)

This notebook implements a complete Group Relative Policy Optimization (GRPO) training pipeline using Hugging Face `trl` and `unsloth` for 4-bit quantization. It connects to the local or remote SOC-Env FastAPI server to collect rollouts and evaluate rewards.

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl transformers accelerate xformers matplotlib vllm requests

In [ ]:
import torch
import requests
import json
import matplotlib.pyplot as plt
from datasets import Dataset
from trl import GRPOTrainer, GRPOConfig
from unsloth import FastLanguageModel

# SOC-Env Connection
SOC_ENV_URL = "http://localhost:7860"  # Update if hosting API elsewhere

### 1. Load Unsloth Model (Qwen2.5-7B-Instruct 4-bit)

In [ ]:
max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### 2. Rollout Collection (reset -> step loop)
Collect `(prompt, response, reward)` tuples to form our training dataset prompt pool. Batching is performed iteratively.

In [ ]:
def collect_rollouts(num_episodes=5):
    dataset_records = []
    for ep in range(num_episodes):
        # Using task_2 for a balance of triage complexity
        resp = requests.post(f"{SOC_ENV_URL}/reset?task=task_2&randomize=true")
        obs = resp.json()
        
        trajectory = []
        done = False
        step = 0
        while not done and step < 10:
            step += 1
            prompt_str = f"CURRENT STATE:\n{json.dumps(obs)}"
            
            # Simulated action (poll_org) to drive the episode and collect trajectory rewards
            action = {"action": "poll_org"}
            step_resp = requests.post(f"{SOC_ENV_URL}/step", json=action).json()
            
            trajectory.append({
                "prompt": prompt_str,
                "step_reward": step_resp["reward"].get("step_reward", 0.0)
            })
            
            obs = step_resp["observation"]
            done = step_resp.get("done", False)
            
            if done:
                # Final scores from the server
                final_reward = step_resp["reward"]
                total_ep_reward = final_reward.get("final_score", 0.0)
                ident = final_reward.get("identification_score", 0.0)
                remed = final_reward.get("remediation_score", 0.0)
                
                for record in trajectory:
                    record["episode_reward"] = total_ep_reward
                    record["identification"] = ident
                    record["remediation"] = remed
                    dataset_records.append(record)
            
    return Dataset.from_list(dataset_records)

train_dataset = collect_rollouts(num_episodes=10)
print(f"Collected {len(train_dataset)} prompts with precomputed episode rewards.")

### 3. TRL GRPOTrainer Initialization & Reward Function
GRPOTrainer will generate multiple completions per prompt. We define a custom reward function that parses the outputs.

In [ ]:
def format_reward_func(completions, episode_reward, **kwargs):
    # Change 1: Rewards are weighted 0.9 x precomputed env reward + 0.1 x JSON bonus
    rewards = []
    for text, ep_rew in zip(completions, episode_reward):
        json_bonus = 0.0
        try:
            # Simple JSON check
            content = text[0]['content'] if isinstance(text, list) else text
            json.loads(content.replace("```json", "").replace("```", "").strip())
            json_bonus = 1.0
        except:
            json_bonus = 0.0
            
        # Combine weighted rewards
        total_reward = (0.9 * float(ep_rew)) + (0.1 * json_bonus)
        rewards.append(total_reward)
    return rewards

def metrics_reward_func(completions, identification, remediation, **kwargs):
    # Tracking metrics for logging (not used for gradients)
    return [float(identification) for _ in completions]

training_args = GRPOConfig(
    output_dir="soc_env_grpo",
    learning_rate=1e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_prompt_length=2000,
    max_completion_length=150,
    num_generations=4,
    max_steps=50,
    logging_steps=5,
)

trainer = GRPOTrainer(
    model=model,
    reward_funcs=[format_reward_func, metrics_reward_func],
    args=training_args,
    train_dataset=train_dataset,
)

In [ ]:
trainer.train()

### 4. Reward Curve Plotting & Inference Traces

In [ ]:
# Change 3: Extract and plot mean rewards, identification, and remediation curves
history = trainer.state.log_history
steps = [h["step"] for h in history if "reward" in h]
# rewards collected by trainer (format_reward_func is the 0th reward)
env_rewards = [h["reward"] for h in history if "reward" in h] 
# metrics_reward_func is the 1st reward
ident_scores = [h.get("reward_metrics_reward_func", 0.0) for h in history if "reward" in h]

# Plotting
plt.figure(figsize=(12, 6))
plt.plot(steps, env_rewards, label="Mean Episode Reward (Weighted)", color="blue", linewidth=2)
plt.plot(steps, ident_scores, label="Mean Identification Score", color="green", linestyle="--")
plt.title("SOC-Env: GRPO Training Progress (Environmental Feedback)")
plt.xlabel("Step")
plt.ylabel("Reward Value")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
FastLanguageModel.for_inference(model)

sample_obs = {"snapshot_type": "org_wide", "devices": [{"device_id": "dev_0"}]}
prompt = f"CURRENT STATE:\n{json.dumps(sample_obs)}"

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=100)
print("Trained Model Output Trace:")
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])